# DataSprint 2.0: D-FINE-M + Top-22% Sky-Crop (Kaggle GPU Twin)

Runs cleanly on free Kaggle GPUs (`GPU P100` 16 GB or `GPU T4 x2` / `T4`) in parallel while Colab L4 finishes `v3`.

## Combined Research & Error-Analysis Architecture
- **Transformer Detector Backbone (`ustc-community/dfine-medium-coco`):** Pure COCO-pretrained (`torch_dtype/dtype=float32`, `do_normalize=False`, `[0,1]` rescale) using Fine-grained Distribution Refinement (FDR) and Global Optimal Localization Self-Distillation (GO-LSD) to cut our #1 TIDE localization bottleneck (`Loc +9.91 mAP@50`).
- **Top-22% Sky-Crop + Exact Coordinate Restore:** Removes top 22% sky/trees (`0.18-0.24` jitter in train, `0.22` fixed at eval/test) based on GRDDC spatial priors and Eigen-CAM background clutter findings (`Bkg +6.99`), zooming road pixels `1.28x` at `640x640` and mapping predictions back via `y += y_off`.
- **India `720x720` LAB-CLAHE + `70/30 OneOf` Robustness Mix:** `p=0.35` L-channel CLAHE during training counters washed-out concrete glare; `30% OneOf` (JPEG 6-22, dark/gamma drop, Gaussian/Motion blur, Gaussian noise) builds corruption robustness.
- **`DamagedGuaranteedBatchSampler`:** Visits 100% of `14,691` training images per epoch with `>=1` damaged image in every batch so D-FINE Contrastive Denoising (`num_denoising=100`) never hits an all-empty batch.
- **4-Way Inference A/B Test + Class-Aware WBF:** Benchmarks raw vs Gamma LUT (`mean_luma < 60 -> 115` for dark DRI blackout) vs bright-concrete CLAHE (`mean_luma > 155`) vs batched Horizontal Flip-TTA + class-aware WBF (preserving cross-class overlaps). Exports `/kaggle/working/submission.csv`, `val_pred_dfine_m.csv`, and `test_pred_dfine_m.csv` for direct ensembling with `v3`.

## Kaggle Quick-Start Checklist
1. Right sidebar -> **Session options -> Accelerator:** choose `GPU P100` (16 GB, fastest single-GPU) or `GPU T4 x2`.
2. Toggle **Internet ON** (needed to download `ustc-community/dfine-medium-coco` weights from Hugging Face).
3. Under **Input**, verify competition dataset `datasprint-2-road-damage-detection` is attached.
4. Click **Run All**.


In [ ]:
import importlib.metadata, re, subprocess, sys

_MIN_VERSIONS = {
    'transformers': (4, 50, 0),
    'albumentations': (2, 0, 0),
    'safetensors': (0, 4, 0),
    'accelerate': (1, 0, 0),
}
_INSTALL_SPECS = {
    'transformers': 'transformers==5.24.0',
    'albumentations': 'albumentations==2.0.8',
    'safetensors': 'safetensors==0.8.0',
    'accelerate': 'accelerate==1.16.0',
}

def _parse_ver_tuple(ver_str: str) -> tuple[int, ...]:
    m = re.match(r'^(\d+(?:\.\d+)*)', str(ver_str).strip())
    return tuple(int(x) for x in m.group(1).split('.')) if m else (0,)

_to_install = []
for pkg, min_v in _MIN_VERSIONS.items():
    try:
        cur_v = importlib.metadata.version(pkg)
        if _parse_ver_tuple(cur_v) < min_v:
            _to_install.append(_INSTALL_SPECS[pkg])
        print(f'{pkg}: installed={cur_v} (min>={".".join(map(str, min_v))})')
    except importlib.metadata.PackageNotFoundError:
        _to_install.append(_INSTALL_SPECS[pkg])
        print(f'{pkg}: NOT INSTALLED -> queueing {_INSTALL_SPECS[pkg]}')

if _to_install:
    print('Installing missing/outdated packages:', _to_install)
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir',
        '--upgrade-strategy', 'only-if-needed', *_to_install,
    ])
else:
    print('All required package versions already satisfied; skipping pip install.')


In [ ]:
import hashlib, json, os, random, shutil, time, uuid
from pathlib import Path
import numpy as np
import torch

STARTED = time.monotonic()
RUN_ID = str(uuid.uuid4())
PROJECT_ROOT = Path('/kaggle/working/dfine_m_output')
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)

CONFIG = json.loads('{\n  "schema_version": "colab-builder-config/1",\n  "project_id": "datasprint-2-0-road-damage",\n  "drive_relative_path": "DataSprint_2.0_Road_Damage",\n  "drive_folder_id": "1n8ddoVvmlpff1ScGLi_WPw-pPBe_AyC1",\n  "confirmed": true,\n  "workload": "training",\n  "packages": [\n    "transformers==5.24.0",\n    "albumentations==2.0.8",\n    "safetensors==0.8.0",\n    "accelerate==1.16.0"\n  ],\n  "dependency_check": "required",\n  "gpu_required": true,\n  "min_gpu_memory_gib": 14,\n  "min_free_disk_gib": 20,\n  "experiment_name": "datasprint2_dfine_m_skycrop22_clahe_oneof30_v1",\n  "budget": {\n    "estimated_hours": 1.8,\n    "max_hours": 6.0,\n    "burn_rate_cu_per_hour": 5.4\n  },\n  "experiment": {\n    "seed": 42,\n    "model_repo": "ustc-community/dfine-medium-coco",\n    "fallback_repo": "ustc-community/dfine-small-coco",\n    "imgsz": 640,\n    "sky_crop_frac": 0.22,\n    "sky_crop_jitter_min": 0.18,\n    "sky_crop_jitter_max": 0.24,\n    "clahe_train_p": 0.35,\n    "robustness_aug_p": 0.3,\n    "epochs": 24,\n    "epochs_t4": 18,\n    "batch_a100": 16,\n    "accum_a100": 2,\n    "batch_l4": 12,\n    "accum_l4": 2,\n    "batch_t4": 8,\n    "accum_t4": 3,\n    "lr_head": 0.0002,\n    "lr_backbone": 2e-05,\n    "weight_decay": 0.0001,\n    "warmup_epochs": 1.5,\n    "max_grad_norm": 0.1,\n    "ema_decay": 0.9998,\n    "eval_every_epochs": 3,\n    "conf_floor": 0.001,\n    "max_det": 100,\n    "lowlight_luma_thresh": 60.0,\n    "lowlight_target_luma": 115.0,\n    "bright_clahe_luma_thresh": 155.0,\n    "test_flip_tta": true,\n    "wbf_iou_thr": 0.55,\n    "smoke_only": false\n  },\n  "context": {\n    "dataset_id": "rdd2020_datasprint_2_0",\n    "dataset_revision": "2026-11-23",\n    "split_id": "official_train_val_test",\n    "split_hash": "train14691_val2150_test4200",\n    "evaluator_id": "official_score_py",\n    "evaluator_revision": "starter_kit_v1",\n    "evaluation_config_hash": "coco101_iou50_95_bestf1_t4speed_skycrop22",\n    "n_samples": 2150,\n    "image_resolution": 640,\n    "iou_threshold": 0.5,\n    "confidence_floor": 0.001,\n    "max_dets_per_image": 100\n  },\n  "metric_context": {\n    "accuracy_score_100": {\n      "unit": "score_0_100",\n      "higher_is_better": true,\n      "split": "val_clean"\n    },\n    "mAP50": {\n      "unit": "ratio_0_1",\n      "higher_is_better": true,\n      "split": "val_clean"\n    },\n    "mAP50_95": {\n      "unit": "ratio_0_1",\n      "higher_is_better": true,\n      "split": "val_clean"\n    },\n    "precision": {\n      "unit": "ratio_0_1",\n      "higher_is_better": true,\n      "split": "val_clean"\n    },\n    "recall": {\n      "unit": "ratio_0_1",\n      "higher_is_better": true,\n      "split": "val_clean"\n    },\n    "ap50_D00": {\n      "unit": "ratio_0_1",\n      "higher_is_better": true,\n      "split": "val_clean"\n    },\n    "ap50_D10": {\n      "unit": "ratio_0_1",\n      "higher_is_better": true,\n      "split": "val_clean"\n    },\n    "ap50_D20": {\n      "unit": "ratio_0_1",\n      "higher_is_better": true,\n      "split": "val_clean"\n    },\n    "ap50_D40": {\n      "unit": "ratio_0_1",\n      "higher_is_better": true,\n      "split": "val_clean"\n    },\n    "speed_score_t4": {\n      "unit": "ratio_0_1",\n      "higher_is_better": true,\n      "split": "val_clean"\n    },\n    "median_ms_t4": {\n      "unit": "ms",\n      "higher_is_better": false,\n      "split": "val_clean"\n    },\n    "robustness_mAP50_synth": {\n      "unit": "ratio_0_1",\n      "higher_is_better": true,\n      "split": "val_degraded_synth"\n    },\n    "final_score_proxy_100": {\n      "unit": "score_0_100",\n      "higher_is_better": true,\n      "split": "val_combined_proxy"\n    },\n    "ab_winner_delta_final_pts": {\n      "unit": "score_delta_pts",\n      "higher_is_better": true,\n      "split": "val_ab_test"\n    },\n    "peak_vram_gib": {\n      "unit": "GiB",\n      "higher_is_better": false,\n      "split": "runtime"\n    },\n    "estimated_compute_units": {\n      "unit": "CU",\n      "higher_is_better": false,\n      "split": "runtime"\n    },\n    "train_images_count": {\n      "unit": "count",\n      "higher_is_better": true,\n      "split": "train"\n    },\n    "val_images_count": {\n      "unit": "count",\n      "higher_is_better": true,\n      "split": "val"\n    },\n    "effective_batch_size": {\n      "unit": "count",\n      "higher_is_better": true,\n      "split": "train"\n    }\n  }\n}')
CHECKPOINT_CONFIG = CONFIG.get('checkpoint', {
    'cadence': 'every_3_epochs_and_final',
    'keep_best': True,
    'format': 'safetensors',
})
TASK_ARTIFACTS = {}
metrics = {}


def check_budget():
    max_h = float(CONFIG.get('budget', {}).get('max_hours', 9.0))
    elapsed_h = (time.monotonic() - STARTED) / 3600.0
    if elapsed_h >= max_h:
        raise RuntimeError(f'Execution budget exceeded: {elapsed_h:.2f}h >= {max_h:.2f}h')


class _KaggleAutopilotHelper:
    def seed_everything(self, seed: int, deterministic: bool = False) -> dict:
        s = int(seed)
        os.environ['PYTHONHASHSEED'] = str(s)
        random.seed(s)
        np.random.seed(s)
        torch.manual_seed(s)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(s)
        torch.backends.cudnn.deterministic = bool(deterministic)
        return {'seed': s, 'deterministic': bool(deterministic)}

    def file_digest(self, path) -> str:
        h = hashlib.sha256()
        with Path(path).open('rb') as f:
            for chunk in iter(lambda: f.read(1048576), b''):
                h.update(chunk)
        return h.hexdigest()

    def save_checkpoint_manifest(self, project_root, run_id: str, name: str, config: dict, rel_files: list) -> Path:
        root = Path(project_root)
        ckpt_dir = root / '03_results' / 'runs' / run_id / 'checkpoints'
        ckpt_dir.mkdir(parents=True, exist_ok=True)
        files_meta = []
        for rel in rel_files:
            fp = root / rel
            if fp.is_file():
                files_meta.append({
                    'path': str(rel),
                    'bytes': int(fp.stat().st_size),
                    'sha256': self.file_digest(fp),
                })
        manifest_path = ckpt_dir / f'{name}_manifest.json'
        payload = {
            'schema': 'checkpoint-manifest/1',
            'run_id': run_id,
            'name': name,
            'checkpoint_config': config,
            'files': files_meta,
            'saved_at_s': round(time.monotonic() - STARTED, 2),
        }
        manifest_path.write_text(json.dumps(payload, indent=2), encoding='utf-8')
        return manifest_path

    def collect_colab_telemetry(self, elapsed_s: float, burn_rate_cu_per_hour=None) -> dict:
        out = {
            'runtime': 'kaggle',
            'elapsed_s': round(float(elapsed_s), 2),
            'estimated_compute_units': 0.0,
        }
        if torch.cuda.is_available():
            out['peak_vram_gib'] = round(torch.cuda.max_memory_allocated(0) / (1024 ** 3), 4)
        return out

    def stage_to_local_cache(self, project_root, rel_path: str, cache_root='/kaggle/working/autopilot_cache') -> dict:
        src = Path(project_root) / rel_path
        dst = Path(cache_root) / rel_path
        dst.parent.mkdir(parents=True, exist_ok=True)
        if src.is_file() and (not dst.is_file() or dst.stat().st_size != src.stat().st_size):
            shutil.copy2(src, dst)
        return {'local_path': str(dst)}


autopilot = _KaggleAutopilotHelper()


def _execute_stage(idx: int, code: str):
    t0 = time.monotonic()
    print(f'=== Starting Stage {idx + 1} (run={RUN_ID}) ===')
    exec(compile(code, f'<stage-{idx + 1}>', 'exec'), globals())
    print(f'=== Completed Stage {idx + 1} in {time.monotonic() - t0:.1f}s ===')


print(f'Kaggle runner initialized | RUN_ID={RUN_ID} | PROJECT_ROOT={PROJECT_ROOT}')


## 1. Preflight, D-FINE-M HF Contract & Precision Dry-Run

Validate ustc-community/dfine-medium-coco (do_normalize=False, [0,1] scale), run a real 2-image (damaged+empty) AMP stability dry-run, detect Colab vs Kaggle runtime, and verify 21,041 unarchived images.


In [ ]:
_execute_stage(0, 'import csv\nimport inspect\nimport json\nimport math\nimport os\nimport random\nimport shutil\nimport subprocess\nimport sys\nimport time\nfrom concurrent.futures import ThreadPoolExecutor\nfrom pathlib import Path\n\nimport albumentations as A\nimport cv2\nimport numpy as np\nimport pandas as pd\nimport torch\nimport torch.nn.functional as F\nimport transformers\nfrom PIL import Image\nfrom torch.utils.data import DataLoader, Dataset, Sampler\nfrom transformers import (\n    AutoImageProcessor,\n    DFineForObjectDetection,\n    get_cosine_schedule_with_warmup,\n)\n\n_TF_MAJOR = int(transformers.__version__.split(\'.\')[0])\n_TF_DTYPE_KW = {\'dtype\': torch.float32} if _TF_MAJOR >= 5 else {\'torch_dtype\': torch.float32}\n\nEXP_CFG = CONFIG[\'experiment\']\nSEED = int(EXP_CFG.get(\'seed\', 42))\nSEED_RECEIPT = autopilot.seed_everything(SEED, deterministic=False)\nCONFIG[\'experiment\'][\'seed_receipt\'] = SEED_RECEIPT\n\nif not torch.cuda.is_available():\n    raise RuntimeError(\'CUDA GPU is required for D-FINE-M training. Select GPU T4/P100/L4/A100.\')\n\n# Ampere/Ada TF32 + cuDNN kernel autotuning\ntorch.backends.cuda.matmul.allow_tf32 = True\ntorch.backends.cudnn.allow_tf32 = True\ntorch.set_float32_matmul_precision(\'high\')\ntorch.backends.cudnn.benchmark = True\n\ndef _avail_ram_gib() -> float:\n    try:\n        return (os.sysconf(\'SC_PAGE_SIZE\') * os.sysconf(\'SC_AVPHYS_PAGES\')) / (1024 ** 3)\n    except Exception:\n        for line in Path(\'/proc/meminfo\').read_text(encoding=\'utf-8\').splitlines():\n            if line.startswith(\'MemAvailable:\'):\n                return float(line.split()[1]) / (1024 ** 2)\n    return 12.0\n\nIS_KAGGLE_ENV = Path(\'/kaggle/input\').is_dir() and not Path(\'/content/drive/MyDrive\').is_dir()\nWORK_BASE = Path(\'/kaggle/working\') if IS_KAGGLE_ENV else Path(\'/content\')\nCACHE_ROOT = WORK_BASE / \'autopilot_cache\'\nHF_CACHE_DIR = CACHE_ROOT / \'hf_cache\'\nHF_CACHE_DIR.mkdir(parents=True, exist_ok=True)\nos.environ[\'HF_HOME\'] = str(HF_CACHE_DIR)\n\nGPU_COUNT = torch.cuda.device_count()\nGPU_NAME = torch.cuda.get_device_name(0)\nGPU_VRAM_GIB = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)\nFREE_RAM_GIB = _avail_ram_gib()\nFREE_DISK_GIB = shutil.disk_usage(str(WORK_BASE)).free / (1024 ** 3)\nCPU_CORES = os.cpu_count() or 4\n\nif GPU_VRAM_GIB >= 36.0:\n    GPU_TIER = \'A100\'\n    BATCH_SIZE = int(EXP_CFG.get(\'batch_a100\', 16))\n    GRAD_ACCUM = int(EXP_CFG.get(\'accum_a100\', 2))\n    EVAL_BATCH = 16\n    DATALOADER_WORKERS = min(8, CPU_CORES)\n    TARGET_EPOCHS = int(EXP_CFG.get(\'epochs\', 24))\nelif GPU_VRAM_GIB >= 20.0:\n    GPU_TIER = \'L4\'\n    BATCH_SIZE = int(EXP_CFG.get(\'batch_l4\', 12))\n    GRAD_ACCUM = int(EXP_CFG.get(\'accum_l4\', 2))\n    EVAL_BATCH = 12\n    DATALOADER_WORKERS = min(6, CPU_CORES)\n    TARGET_EPOCHS = int(EXP_CFG.get(\'epochs\', 24))\nelse:\n    GPU_TIER = \'T4\'\n    BATCH_SIZE = int(EXP_CFG.get(\'batch_t4\', 8))\n    GRAD_ACCUM = int(EXP_CFG.get(\'accum_t4\', 3))\n    EVAL_BATCH = 8\n    DATALOADER_WORKERS = min(4, CPU_CORES)\n    TARGET_EPOCHS = int(EXP_CFG.get(\'epochs_t4\', 18))\n\nif EXP_CFG.get(\'smoke_only\'):\n    TARGET_EPOCHS = 1\n\nprint(f\'Env: {"Kaggle" if IS_KAGGLE_ENV else "Colab"} | PyTorch: {torch.__version__} | GPUs: {GPU_COUNT}x {GPU_NAME} ({GPU_VRAM_GIB:.1f} GiB -> {GPU_TIER})\')\nprint(f\'RAM free: {FREE_RAM_GIB:.1f} GiB | Disk free: {FREE_DISK_GIB:.1f} GiB | CPUs: {CPU_CORES}\')\nprint(f\'Auto config -> bs={BATCH_SIZE}, grad_accum={GRAD_ACCUM} (eff={BATCH_SIZE * GRAD_ACCUM}), epochs={TARGET_EPOCHS}, workers={DATALOADER_WORKERS}\')\n\nCLASS_MAP = {\n    0: \'D00_longitudinal_crack\',\n    1: \'D10_transverse_crack\',\n    2: \'D20_alligator_crack\',\n    3: \'D40_pothole\',\n}\nLABEL2ID = {v: k for k, v in CLASS_MAP.items()}\nMODEL_REPO = str(EXP_CFG.get(\'model_repo\', \'ustc-community/dfine-medium-coco\'))\nIMGSZ = int(EXP_CFG.get(\'imgsz\', 640))\nSKY_CROP_FRAC = float(EXP_CFG.get(\'sky_crop_frac\', 0.22))\n\n# Verify HF D-FINE preprocessor contract & run real 2-image dry-run (damaged + empty 0-box)\nassert \'ignore_mismatched_sizes\' in inspect.signature(DFineForObjectDetection.from_pretrained).parameters\nIMAGE_PROCESSOR = AutoImageProcessor.from_pretrained(MODEL_REPO, cache_dir=str(HF_CACHE_DIR))\nprint(f\'HF ImageProcessor: {type(IMAGE_PROCESSOR).__name__} | size={IMAGE_PROCESSOR.size} | do_rescale={getattr(IMAGE_PROCESSOR, "do_rescale", None)} | do_normalize={getattr(IMAGE_PROCESSOR, "do_normalize", None)}\')\nassert getattr(IMAGE_PROCESSOR, \'do_normalize\', False) is False, \'D-FINE preprocessor expects [0,1] rescale without ImageNet mean/std normalization\'\nassert getattr(IMAGE_PROCESSOR, \'do_rescale\', True) is True\n\n# Select safe AMP precision and validate via real forward + backward on [damaged, empty] batch\nDEVICE = torch.device(\'cuda:0\')\n_candidate_amp = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16\n_dry_model = DFineForObjectDetection.from_pretrained(\n    MODEL_REPO,\n    id2label=CLASS_MAP,\n    label2id=LABEL2ID,\n    ignore_mismatched_sizes=True,\n    cache_dir=str(HF_CACHE_DIR),\n    **_TF_DTYPE_KW,\n).to(DEVICE)\n_dry_model.train()\n\n_dry_px = torch.rand((2, 3, IMGSZ, IMGSZ), device=DEVICE, dtype=torch.float32)\n_dry_labels = [\n    {\'class_labels\': torch.tensor([0, 2], device=DEVICE, dtype=torch.long), \'boxes\': torch.tensor([[0.5, 0.6, 0.2, 0.3], [0.3, 0.7, 0.25, 0.2]], device=DEVICE, dtype=torch.float32)},\n    {\'class_labels\': torch.zeros((0,), device=DEVICE, dtype=torch.long), \'boxes\': torch.zeros((0, 4), device=DEVICE, dtype=torch.float32)},\n]\n\n# Cross-check annotation encoding vs IMAGE_PROCESSOR reference\n_ref_ann = {\'image_id\': 0, \'annotations\': [{\'id\': 0, \'image_id\': 0, \'category_id\': 0, \'bbox\': [256.0, 288.0, 128.0, 192.0], \'area\': 24576.0, \'iscrowd\': 0}]}\n_ref_proc = IMAGE_PROCESSOR(images=np.zeros((IMGSZ, IMGSZ, 3), dtype=np.uint8), annotations=_ref_ann, return_tensors=\'pt\')\n_ref_box = _ref_proc[\'labels\'][0][\'boxes\'][0].numpy()\nassert np.allclose(_ref_box, np.array([0.5, 0.6, 0.2, 0.3], dtype=np.float32), atol=1e-5), f\'Box format mismatch: {_ref_box}\'\n\nAMP_DTYPE = _candidate_amp\ntry:\n    with torch.amp.autocast(\'cuda\', dtype=_candidate_amp, enabled=True):\n        _dry_out = _dry_model(pixel_values=_dry_px, labels=_dry_labels)\n    if not torch.isfinite(_dry_out.loss):\n        raise FloatingPointError(\'Non-finite AMP dry-run loss\')\n    _dry_out.loss.backward()\n    for _p in _dry_model.parameters():\n        if _p.grad is not None and not torch.isfinite(_p.grad).all():\n            raise FloatingPointError(\'Non-finite AMP gradient\')\n    with torch.no_grad():\n        _dry_out.logits = _dry_out.logits.detach().float()\n        _dry_out.pred_boxes = _dry_out.pred_boxes.detach().float()\n        _dry_dec = IMAGE_PROCESSOR.post_process_object_detection(\n            _dry_out,\n            target_sizes=torch.tensor([[IMGSZ, IMGSZ], [IMGSZ, IMGSZ]], dtype=torch.long),\n            threshold=0.001,\n        )\n        assert _dry_dec[0][\'scores\'].detach().float().cpu().numpy().dtype == np.float32\n        assert _dry_dec[0][\'boxes\'].detach().float().cpu().numpy().dtype == np.float32\n    print(f\'D-FINE-M real 2-image dry-run PASSED with AMP dtype={AMP_DTYPE} (loss={float(_dry_out.loss.detach()):.4f}).\')\nexcept Exception as _amp_exc:\n    print(f\'WARNING: AMP {_candidate_amp} unstable on {GPU_NAME} ({_amp_exc}); falling back to FP32 for guaranteed convergence.\')\n    AMP_DTYPE = None\n    _dry_model.zero_grad(set_to_none=True)\n    _dry_out_fp32 = _dry_model(pixel_values=_dry_px, labels=_dry_labels)\n    assert torch.isfinite(_dry_out_fp32.loss), \'FP32 dry-run loss non-finite!\'\n    _dry_out_fp32.loss.backward()\n    print(f\'D-FINE-M FP32 fallback dry-run PASSED (loss={float(_dry_out_fp32.loss.detach()):.4f}).\')\n\ndel _dry_model, _dry_px, _dry_labels\ntorch.cuda.empty_cache()\n\n# Auto-resolve competition dataset directory (Colab Drive vs Kaggle /kaggle/input)\nif IS_KAGGLE_ENV:\n    _cands = list(Path(\'/kaggle/input\').rglob(\'train_labels.csv\'))\n    if not _cands:\n        raise FileNotFoundError(\'Could not locate train_labels.csv under /kaggle/input. Attach datasprint-2-road-damage-detection dataset.\')\n    DRIVE_DATA_DIR = _cands[0].parent\n    STARTER_SRC_DIR = DRIVE_DATA_DIR\nelse:\n    DRIVE_DATA_DIR = PROJECT_ROOT / \'01_data\'\n    STARTER_SRC_DIR = PROJECT_ROOT / \'02_notebooks\' / \'starter_kit\'\n\nRUN_DRIVE_DIR = PROJECT_ROOT / \'03_results\' / \'runs\' / RUN_ID\nRUN_DRIVE_DIR.mkdir(parents=True, exist_ok=True)\n(RUN_DRIVE_DIR / \'checkpoints\').mkdir(parents=True, exist_ok=True)\n(RUN_DRIVE_DIR / \'weights\').mkdir(parents=True, exist_ok=True)\n\nrequired_csvs = [\'train_labels.csv\', \'val_labels.csv\', \'val_image_ids.csv\', \'test_image_ids.csv\']\nfor csv_name in required_csvs:\n    if not (DRIVE_DATA_DIR / csv_name).is_file():\n        raise FileNotFoundError(f\'Missing {csv_name} in {DRIVE_DATA_DIR}\')\n\ndef _count_jpgs(folder: Path) -> int:\n    return sum(1 for p in folder.iterdir() if p.is_file() and p.suffix.lower() in {\'.jpg\', \'.jpeg\'}) if folder.is_dir() else 0\n\ntrain_count = _count_jpgs(DRIVE_DATA_DIR / \'train_images\')\nval_count = _count_jpgs(DRIVE_DATA_DIR / \'val_images\')\ntest_count = _count_jpgs(DRIVE_DATA_DIR / \'test_images\')\nprint(f\'Source image counts -> train: {train_count}, val: {val_count}, test: {test_count}\')\nif train_count < 14691 or val_count < 2150 or test_count < 4200:\n    raise RuntimeError(f\'Expected full dataset (14691/2150/4200), got train={train_count}, val={val_count}, test={test_count}\')')


## 2. Parallel Local SSD Staging & Dataset Integrity Summary

Stage all 4 CSVs, 6 starter scripts, and 21,041 images to fast local storage (parallel copy on Colab / instant symlinks on Kaggle) and log eda_summary.json.


In [ ]:
_execute_stage(1, "KIT_LOCAL = CACHE_ROOT / 'datasprint_kit'\nKIT_LOCAL.mkdir(parents=True, exist_ok=True)\n\nfor csv_name in required_csvs:\n    src_csv = DRIVE_DATA_DIR / csv_name\n    if IS_KAGGLE_ENV:\n        shutil.copy2(src_csv, KIT_LOCAL / csv_name)\n    else:\n        rec = autopilot.stage_to_local_cache(PROJECT_ROOT, f'01_data/{csv_name}', cache_root=str(CACHE_ROOT))\n        shutil.copy2(rec['local_path'], KIT_LOCAL / csv_name)\n\nfor sname in ('benchmark_speed.py', 'check_submission.py', 'inference.py', 'make_kaggle_submission.py', 'score.py', 'train_baseline.py'):\n    if IS_KAGGLE_ENV:\n        cand_s = STARTER_SRC_DIR / sname\n        if not cand_s.is_file():\n            found_s = list(Path('/kaggle/input').rglob(sname))\n            cand_s = found_s[0] if found_s else cand_s\n        shutil.copy2(cand_s, KIT_LOCAL / sname)\n    else:\n        s_rec = autopilot.stage_to_local_cache(PROJECT_ROOT, f'02_notebooks/starter_kit/{sname}', cache_root=str(CACHE_ROOT))\n        shutil.copy2(s_rec['local_path'], KIT_LOCAL / sname)\n\ndef _stage_split_fast(split_folder: str) -> int:\n    src_dir = DRIVE_DATA_DIR / split_folder\n    dst_dir = KIT_LOCAL / split_folder\n    dst_dir.mkdir(parents=True, exist_ok=True)\n    files = [p for p in src_dir.iterdir() if p.is_file() and p.suffix.lower() in {'.jpg', '.jpeg'}]\n    t0 = time.monotonic()\n    if IS_KAGGLE_ENV:\n        for p in files:\n            tgt = dst_dir / p.name\n            if not tgt.exists():\n                os.symlink(p.resolve(), tgt)\n        print(f'  [{split_folder}] linked {len(files)} images in {time.monotonic() - t0:.2f}s')\n        return len(files)\n\n    def _copy_one(p: Path):\n        tgt = dst_dir / p.name\n        if not tgt.is_file() or tgt.stat().st_size != p.stat().st_size:\n            shutil.copy2(p, tgt)\n        return 1\n\n    done = 0\n    with ThreadPoolExecutor(max_workers=24) as pool:\n        for _ in pool.map(_copy_one, files, chunksize=64):\n            done += 1\n            if done % 3000 == 0 or done == len(files):\n                print(f'  [{split_folder}] staged {done}/{len(files)} ({done / max(0.01, time.monotonic() - t0):.0f} img/s)')\n    return len(files)\n\nstaged_train = _stage_split_fast('train_images')\nstaged_val = _stage_split_fast('val_images')\nstaged_test = _stage_split_fast('test_images')\ncheck_budget()\n\ntrain_df = pd.read_csv(KIT_LOCAL / 'train_labels.csv')\nval_df = pd.read_csv(KIT_LOCAL / 'val_labels.csv')\nval_ids_df = pd.read_csv(KIT_LOCAL / 'val_image_ids.csv')\ntest_ids_df = pd.read_csv(KIT_LOCAL / 'test_image_ids.csv')\n\ndef _vec_split_summary(df: pd.DataFrame, total_imgs: int) -> dict:\n    n_dam = int(df['image_id'].nunique())\n    vc = df['class_id'].value_counts().sort_index()\n    return {\n        'total_images': int(total_imgs),\n        'damaged_images': n_dam,\n        'background_only_images': int(total_imgs - n_dam),\n        'total_boxes': int(len(df)),\n        'invalid_boxes': int(((df['x_max'] <= df['x_min']) | (df['y_max'] <= df['y_min'])).sum()),\n        'class_counts': {CLASS_MAP[int(k)]: int(v) for k, v in vc.items()},\n    }\n\neda_summary = {\n    'train': _vec_split_summary(train_df, staged_train),\n    'val': _vec_split_summary(val_df, staged_val),\n    'val_image_ids_rows': int(len(val_ids_df)),\n    'test_image_ids_rows': int(len(test_ids_df)),\n    'staged_test_images': int(staged_test),\n}\n(RUN_DRIVE_DIR / 'eda_summary.json').write_text(json.dumps(eda_summary, indent=2), encoding='utf-8')\nTASK_ARTIFACTS['eda_summary'] = f'03_results/runs/{RUN_ID}/eda_summary.json'\nprint(json.dumps(eda_summary, indent=2))")


## 3. Top-22% Sky-Crop Audit, LAB-CLAHE + 70/30 OneOf Dataset & 4-Bucket Stress Val

Synthesizes GRDDC Top-22% sky-crop, Eigen-CAM background cleanup, India 720x720 bright-concrete LAB-CLAHE, 70/30 OneOf robustness augs, DamagedGuaranteedBatchSampler, and deterministic 4-bucket stress validation.


In [ ]:
_execute_stage(2, 'WORK_ROOT = CACHE_ROOT / \'dfine_work\' / RUN_ID\nWORK_ROOT.mkdir(parents=True, exist_ok=True)\n\n# 1. Core geometric/photometric helpers (shared across train, eval & standalone inference)\ndef compute_sky_crop_offset(img_h: int, crop_frac: float) -> int:\n    return max(0, min(img_h - 32, int(round(float(img_h) * float(crop_frac)))))\n\ndef shift_boxes_for_sky_crop(xyxy_list, labels_list, img_h: int, y_off: int, min_h: float = 4.0, min_area_vis: float = 0.20):\n    crop_h = float(img_h - y_off)\n    out_boxes, out_labels = [], []\n    for box, cls_id in zip(xyxy_list, labels_list):\n        x1, y1, x2, y2 = float(box[0]), float(box[1]), float(box[2]), float(box[3])\n        old_area = max(1e-6, (x2 - x1) * (y2 - y1))\n        ny1 = max(0.0, y1 - float(y_off))\n        ny2 = min(crop_h, y2 - float(y_off))\n        nw = max(0.0, x2 - x1)\n        nh = max(0.0, ny2 - ny1)\n        if nh < min_h or nw < 2.0 or (nw * nh) / old_area < min_area_vis:\n            continue\n        out_boxes.append([x1, ny1, x2, ny2])\n        out_labels.append(int(cls_id))\n    return out_boxes, out_labels\n\ndef restore_boxes_from_sky_crop(xyxy_arr: np.ndarray, orig_h: int, orig_w: int, y_off: int) -> np.ndarray:\n    if len(xyxy_arr) == 0:\n        return xyxy_arr\n    out = xyxy_arr.copy()\n    out[:, 0] = np.clip(out[:, 0], 0.0, float(orig_w))\n    out[:, 1] = np.clip(out[:, 1] + float(y_off), 0.0, float(orig_h))\n    out[:, 2] = np.clip(out[:, 2], 0.0, float(orig_w))\n    out[:, 3] = np.clip(out[:, 3] + float(y_off), 0.0, float(orig_h))\n    return out\n\ndef apply_lab_clahe_rgb(img_rgb: np.ndarray, clip_limit: float = 2.0, grid_size: int = 8) -> np.ndarray:\n    lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)\n    l_chan, a_chan, b_chan = cv2.split(lab)\n    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=(grid_size, grid_size))\n    l_eq = clahe.apply(l_chan)\n    merged = cv2.merge((l_eq, a_chan, b_chan))\n    return cv2.cvtColor(merged, cv2.COLOR_LAB2RGB)\n\ndef build_lut_table(luma_thresh: float, target_luma: float) -> dict:\n    x = np.arange(256, dtype=np.float32) / 255.0\n    tgt = math.log(float(target_luma) / 255.0)\n    luts = {}\n    for lum in range(int(luma_thresh)):\n        gamma = max(0.35, min(0.85, tgt / math.log(max(float(lum), 8.0) / 255.0)))\n        luts[lum] = np.clip(np.round(255.0 * (x ** gamma)), 0, 255).astype(np.uint8)\n    return luts\n\ndef wbf_class_aware_two_views(boxes_a, scores_a, labels_a, boxes_b, scores_b, labels_b, iou_thr: float = 0.55, max_det: int = 100):\n    """Class-aware Weighted Boxes Fusion across two TTA views (preserves cross-class overlaps)."""\n    merged = []\n    all_classes = sorted(set([int(c) for c in labels_a] + [int(c) for c in labels_b]))\n    for cls_id in all_classes:\n        cand = []\n        for idx_b, (b, s, c) in enumerate(zip(boxes_a, scores_a, labels_a)):\n            if int(c) == cls_id:\n                cand.append((float(s), np.asarray(b, dtype=np.float64), 0))\n        for idx_b, (b, s, c) in enumerate(zip(boxes_b, scores_b, labels_b)):\n            if int(c) == cls_id:\n                cand.append((float(s), np.asarray(b, dtype=np.float64), 1))\n        cand.sort(key=lambda t: t[0], reverse=True)\n        clusters = []\n        for sc, box, view_id in cand:\n            best_i = -1\n            best_iou = iou_thr\n            bx1, by1, bx2, by2 = box\n            barea = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)\n            for ci, cl in enumerate(clusters):\n                cx1, cy1, cx2, cy2 = cl[\'fused\']\n                iw = max(0.0, min(bx2, cx2) - max(bx1, cx1))\n                ih = max(0.0, min(by2, cy2) - max(by1, cy1))\n                inter = iw * ih\n                if inter <= 0.0:\n                    continue\n                carea = max(0.0, cx2 - cx1) * max(0.0, cy2 - cy1)\n                iou = inter / max(1e-9, barea + carea - inter)\n                if iou >= best_iou:\n                    best_iou = iou\n                    best_i = ci\n            if best_i >= 0:\n                cl = clusters[best_i]\n                w_sum = cl[\'w_sum\'] + sc\n                cl[\'fused\'] = (cl[\'fused\'] * cl[\'w_sum\'] + box * sc) / max(1e-9, w_sum)\n                cl[\'w_sum\'] = w_sum\n                cl[\'scores\'].append(sc)\n                cl[\'views\'].add(view_id)\n            else:\n                clusters.append({\'fused\': box.copy(), \'w_sum\': sc, \'scores\': [sc], \'views\': {view_id}})\n        for cl in clusters:\n            # Penalize single-view-only boxes slightly by N_views / 2.0 (standard WBF with 2 views)\n            fused_sc = float(sum(cl[\'scores\']) / 2.0)\n            fb = cl[\'fused\']\n            merged.append((cls_id, fused_sc, float(fb[0]), float(fb[1]), float(fb[2]), float(fb[3])))\n    merged.sort(key=lambda t: t[1], reverse=True)\n    return merged[:max_det]\n\n# 2. Vectorized Sky-Crop Retention Audit on all 20,078 GT boxes\ndef _build_image_index(split_name: str, lbl_df: pd.DataFrame):\n    img_dir = KIT_LOCAL / f\'{split_name}_images\'\n    img_paths = sorted(img_dir.glob(\'*.jpg\'))\n    def _read_dim(p: Path):\n        with Image.open(p) as im:\n            return p.stem, im.size[0], im.size[1], str(p)\n    with ThreadPoolExecutor(max_workers=16) as pool:\n        rows = list(pool.map(_read_dim, img_paths, chunksize=64))\n    dims_df = pd.DataFrame(rows, columns=[\'image_id\', \'w\', \'h\', \'path\'])\n    grouped = {}\n    for iid, grp in lbl_df.groupby(\'image_id\'):\n        grouped[iid] = (\n            grp[[\'x_min\', \'y_min\', \'x_max\', \'y_max\']].to_numpy(dtype=np.float32).tolist(),\n            grp[\'class_id\'].to_numpy(dtype=np.int64).tolist(),\n        )\n    records = []\n    n_total_boxes, n_kept_boxes, n_clipped_boxes = 0, 0, 0\n    for row in rows:\n        iid, w, h, p_str = row\n        boxes, labels = grouped.get(iid, ([], []))\n        y_off = compute_sky_crop_offset(h, SKY_CROP_FRAC)\n        kb, _ = shift_boxes_for_sky_crop(boxes, labels, h, y_off)\n        n_total_boxes += len(boxes)\n        n_kept_boxes += len(kb)\n        for bx in boxes:\n            if float(bx[1]) < float(y_off) < float(bx[3]):\n                n_clipped_boxes += 1\n        records.append({\'image_id\': iid, \'w\': int(w), \'h\': int(h), \'path\': p_str, \'boxes\': boxes, \'labels\': labels})\n    pct_kept = 100.0 * n_kept_boxes / max(1, n_total_boxes)\n    print(f\'[{split_name}] Top-{int(SKY_CROP_FRAC*100)}% Sky-Crop Audit: {len(records)} imgs | GT kept {n_kept_boxes}/{n_total_boxes} ({pct_kept:.2f}%), upper-clipped {n_clipped_boxes}\')\n    return records, dims_df\n\ntrain_records, train_dims_df = _build_image_index(\'train\', train_df)\nval_records, val_dims_df = _build_image_index(\'val\', val_df)\n\n# 3. Road-Safe + EDA-Calibrated Dataset & DamagedGuaranteedBatchSampler\nROB_P = float(EXP_CFG.get(\'robustness_aug_p\', 0.30))\nCLAHE_TRAIN_P = float(EXP_CFG.get(\'clahe_train_p\', 0.35))\nCROP_JITTER_MIN = float(EXP_CFG.get(\'sky_crop_jitter_min\', 0.18))\nCROP_JITTER_MAX = float(EXP_CFG.get(\'sky_crop_jitter_max\', 0.24))\n\nclass RoadDamageDFineDataset(Dataset):\n    def __init__(self, records, imgsz: int, is_train: bool = True, override_dir: Path = None):\n        self.records = records\n        self.imgsz = int(imgsz)\n        self.is_train = bool(is_train)\n        self.override_dir = override_dir\n        if self.is_train:\n            self.geom_color_aug = A.Compose([\n                A.HorizontalFlip(p=0.5),\n                A.Affine(scale=(0.88, 1.12), translate_percent={\'x\': (-0.05, 0.05), \'y\': (-0.04, 0.04)}, rotate=0.0, border_mode=cv2.BORDER_CONSTANT, p=0.45),\n                A.ColorJitter(brightness=0.18, contrast=0.18, saturation=0.25, hue=0.015, p=0.45),\n                A.OneOf([\n                    A.ImageCompression(compression_type=\'jpeg\', quality_range=(6, 22), p=1.0),\n                    A.Compose([\n                        A.RandomBrightnessContrast(brightness_limit=(-0.75, -0.45), contrast_limit=(-0.35, 0.0), p=1.0),\n                        A.RandomGamma(gamma_limit=(140, 220), p=0.5),\n                    ], p=1.0),\n                    A.OneOf([\n                        A.GaussianBlur(blur_limit=(7, 15), p=1.0),\n                        A.MotionBlur(blur_limit=(7, 15), p=1.0),\n                    ], p=1.0),\n                    A.GaussNoise(std_range=(0.05, 0.15), p=1.0),\n                ], p=ROB_P),\n            ], bbox_params=A.BboxParams(format=\'pascal_voc\', label_fields=[\'class_labels\'], clip=True, min_area=12.0))\n        else:\n            self.geom_color_aug = None\n\n    def __len__(self) -> int:\n        return len(self.records)\n\n    def __getitem__(self, idx: int):\n        rec = self.records[idx]\n        img_path = (self.override_dir / f"{rec[\'image_id\']}.jpg") if self.override_dir is not None else Path(rec[\'path\'])\n        with Image.open(img_path) as im:\n            img = np.asarray(im.convert(\'RGB\'))\n        orig_h, orig_w = img.shape[0], img.shape[1]\n\n        frac = random.uniform(CROP_JITTER_MIN, CROP_JITTER_MAX) if self.is_train else SKY_CROP_FRAC\n        y_off = compute_sky_crop_offset(orig_h, frac)\n        crop = img[y_off:, :, :]\n        crop_h, crop_w = crop.shape[0], crop.shape[1]\n\n        boxes_px, labels = shift_boxes_for_sky_crop(rec[\'boxes\'], rec[\'labels\'], orig_h, y_off)\n\n        if self.is_train:\n            if random.random() < CLAHE_TRAIN_P:\n                crop = apply_lab_clahe_rgb(crop, clip_limit=2.0, grid_size=8)\n            aug = self.geom_color_aug(image=crop, bboxes=boxes_px, class_labels=labels)\n            crop = aug[\'image\']\n            boxes_px = [list(b) for b in aug[\'bboxes\']]\n            labels = [int(c) for c in aug[\'class_labels\']]\n\n        resized = cv2.resize(crop, (self.imgsz, self.imgsz), interpolation=cv2.INTER_LINEAR)\n        px_tensor = torch.from_numpy(np.ascontiguousarray(resized)).permute(2, 0, 1).to(torch.float32).div_(255.0)\n\n        if boxes_px:\n            arr = np.asarray(boxes_px, dtype=np.float32)\n            # Normalized (cx, cy, w, h) relative to cropped dimensions (crop_w, crop_h)\n            cx = np.clip((arr[:, 0] + arr[:, 2]) * 0.5 / float(crop_w), 0.0, 1.0)\n            cy = np.clip((arr[:, 1] + arr[:, 3]) * 0.5 / float(crop_h), 0.0, 1.0)\n            bw = np.clip((arr[:, 2] - arr[:, 0]) / float(crop_w), 1e-4, 1.0)\n            bh = np.clip((arr[:, 3] - arr[:, 1]) / float(crop_h), 1e-4, 1.0)\n            norm_boxes = torch.from_numpy(np.stack([cx, cy, bw, bh], axis=1)).to(torch.float32)\n            cls_tensor = torch.tensor(labels, dtype=torch.long)\n        else:\n            norm_boxes = torch.zeros((0, 4), dtype=torch.float32)\n            cls_tensor = torch.zeros((0,), dtype=torch.long)\n\n        return {\n            \'pixel_values\': px_tensor,\n            \'target\': {\'class_labels\': cls_tensor, \'boxes\': norm_boxes},\n            \'image_id\': rec[\'image_id\'],\n            \'orig_size\': (orig_h, orig_w),\n            \'crop_size\': (crop_h, crop_w),\n            \'y_off\': y_off,\n        }\n\ndef dfine_collate_fn(batch):\n    return {\n        \'pixel_values\': torch.stack([b[\'pixel_values\'] for b in batch], dim=0),\n        \'labels\': [b[\'target\'] for b in batch],\n        \'image_ids\': [b[\'image_id\'] for b in batch],\n        \'orig_sizes\': [b[\'orig_size\'] for b in batch],\n        \'crop_sizes\': [b[\'crop_size\'] for b in batch],\n        \'y_offs\': [b[\'y_off\'] for b in batch],\n    }\n\nclass DamagedGuaranteedBatchSampler(Sampler):\n    """Visits 100% of train images once per epoch while guaranteeing >=1 damaged image in every batch."""\n    def __init__(self, damaged_indices: list, clean_indices: list, batch_size: int, seed: int = 42):\n        self.damaged = list(damaged_indices)\n        self.clean = list(clean_indices)\n        self.batch_size = int(batch_size)\n        self.seed = int(seed)\n        self.epoch = 0\n        self.total = len(self.damaged) + len(self.clean)\n        self.num_batches = max(1, math.ceil(self.total / self.batch_size))\n        if len(self.damaged) < self.num_batches:\n            raise ValueError(f\'Need damaged >= num_batches ({len(self.damaged)} < {self.num_batches})\')\n\n    def set_epoch(self, ep: int):\n        self.epoch = int(ep)\n\n    def __len__(self) -> int:\n        return self.num_batches\n\n    def __iter__(self):\n        rng = random.Random(self.seed + self.epoch * 9973)\n        dam = list(self.damaged)\n        cln = list(self.clean)\n        rng.shuffle(dam)\n        rng.shuffle(cln)\n        # Anchor 1 guaranteed damaged image into every single batch (including last batch)\n        anchors = dam[:self.num_batches]\n        rest = dam[self.num_batches:] + cln\n        rng.shuffle(rest)\n        batches = []\n        cursor = 0\n        for b_idx in range(self.num_batches):\n            cap = self.batch_size - 1\n            if b_idx == self.num_batches - 1:\n                take = len(rest) - cursor\n            else:\n                take = min(cap, max(0, len(rest) - cursor))\n            batch = [anchors[b_idx]] + rest[cursor:cursor + take]\n            cursor += take\n            rng.shuffle(batch)\n            batches.append(batch)\n        return iter(batches)\n\ntrain_ds = RoadDamageDFineDataset(train_records, IMGSZ, is_train=True)\nval_ds = RoadDamageDFineDataset(val_records, IMGSZ, is_train=False)\ndamaged_idx = [i for i, r in enumerate(train_records) if len(r[\'boxes\']) > 0]\nclean_idx = [i for i, r in enumerate(train_records) if len(r[\'boxes\']) == 0]\ntrain_sampler = DamagedGuaranteedBatchSampler(damaged_idx, clean_idx, BATCH_SIZE, seed=SEED)\ntrain_loader = DataLoader(train_ds, batch_sampler=train_sampler, num_workers=DATALOADER_WORKERS, pin_memory=True, persistent_workers=(DATALOADER_WORKERS > 0), collate_fn=dfine_collate_fn)\nval_loader = DataLoader(val_ds, batch_size=EVAL_BATCH, shuffle=False, num_workers=DATALOADER_WORKERS, pin_memory=True, collate_fn=dfine_collate_fn)\nprint(f\'DataLoaders ready -> train batches={len(train_loader)} (damaged={len(damaged_idx)}, clean={len(clean_idx)}), val batches={len(val_loader)}\')\n\n# 4. Build exact deterministic 4-bucket degraded val set matching v3\nVAL_DEGRADED_DIR = WORK_ROOT / \'val_images_degraded\'\nVAL_DEGRADED_DIR.mkdir(parents=True, exist_ok=True)\nval_jpgs = sorted((KIT_LOCAL / \'val_images\').glob(\'*.jpg\'))\n\n_STRESS_BUCKETS = [\n    lambda s: A.Compose([A.ImageCompression(compression_type=\'jpeg\', quality_range=(8, 18), p=1.0)], seed=s),\n    lambda s: A.Compose([A.RandomBrightnessContrast(brightness_limit=(-0.72, -0.48), contrast_limit=(-0.30, 0.0), p=1.0)], seed=s),\n    lambda s: A.Compose([A.OneOf([A.GaussianBlur(blur_limit=(9, 13), p=1.0), A.MotionBlur(blur_limit=(9, 13), p=1.0)], p=1.0)], seed=s),\n    lambda s: A.Compose([A.GaussNoise(std_range=(0.06, 0.13), p=1.0)], seed=s),\n]\n\ndef _degrade_one(idx_path):\n    idx, p = idx_path\n    out_p = VAL_DEGRADED_DIR / p.name\n    if out_p.is_file():\n        return 1\n    pipe = _STRESS_BUCKETS[idx % 4](SEED + idx)\n    arr = np.asarray(Image.open(p).convert(\'RGB\'))\n    deg = pipe(image=arr)[\'image\']\n    Image.fromarray(deg).save(out_p, format=\'JPEG\', quality=88)\n    return 1\n\nwith ThreadPoolExecutor(max_workers=8) as pool:\n    list(pool.map(_degrade_one, enumerate(val_jpgs), chunksize=32))\nprint(f\'Built 4-bucket degraded val set ({len(val_jpgs)} imgs) at {VAL_DEGRADED_DIR}\')\ncheck_budget()')


## 4. Train D-FINE-M (Differential LR, Warmup+Cosine, EMA, Official Val & Resume)

Fine-tune COCO D-FINE-M with backbone/head differential AdamW LRs, ModelEMA (0.9998), grad clipping (0.1), official score.py validation checkpoints, and full resumable state.


In [ ]:
_execute_stage(3, 'class ModelEMA:\n    def __init__(self, model: torch.nn.Module, decay: float = 0.9998):\n        self.decay = float(decay)\n        self.updates = 0\n        self.shadow = {k: v.detach().clone() for k, v in model.state_dict().items()}\n        self.backup = None\n\n    @torch.no_grad()\n    def update(self, model: torch.nn.Module):\n        self.updates += 1\n        d = self.decay * (1.0 - math.exp(-self.updates / 2000.0))\n        msd = model.state_dict()\n        for k, v in msd.items():\n            if not torch.is_floating_point(v):\n                self.shadow[k].copy_(v)\n            else:\n                self.shadow[k].mul_(d).add_(v.detach(), alpha=1.0 - d)\n\n    def apply_shadow(self, model: torch.nn.Module):\n        self.backup = {k: v.detach().clone() for k, v in model.state_dict().items()}\n        model.load_state_dict(self.shadow, strict=True)\n\n    def restore(self, model: torch.nn.Module):\n        if self.backup is not None:\n            model.load_state_dict(self.backup, strict=True)\n            self.backup = None\n\n@torch.no_grad()\ndef _evaluate_dfine_to_csv(eval_model, loader, out_csv: Path, conf_floor: float = 0.001, max_det: int = 100):\n    eval_model.eval()\n    rows = []\n    for batch in loader:\n        px = batch[\'pixel_values\'].to(DEVICE, non_blocking=True)\n        with torch.amp.autocast(\'cuda\', dtype=AMP_DTYPE, enabled=(AMP_DTYPE is not None)):\n            outputs = eval_model(pixel_values=px)\n        outputs.logits = outputs.logits.float()\n        outputs.pred_boxes = outputs.pred_boxes.float()\n        crop_sizes = torch.tensor(batch[\'crop_sizes\'], dtype=torch.long)\n        decoded = IMAGE_PROCESSOR.post_process_object_detection(outputs, target_sizes=crop_sizes, threshold=conf_floor)\n        for iid, orig_sz, y_off, res in zip(batch[\'image_ids\'], batch[\'orig_sizes\'], batch[\'y_offs\'], decoded):\n            scores = res[\'scores\'].detach().float().cpu().numpy()\n            labels = res[\'labels\'].detach().cpu().numpy().astype(int)\n            boxes = res[\'boxes\'].detach().float().cpu().numpy()\n            if len(scores) > max_det:\n                keep = np.argsort(-scores)[:max_det]\n                scores, labels, boxes = scores[keep], labels[keep], boxes[keep]\n            boxes = restore_boxes_from_sky_crop(boxes, orig_sz[0], orig_sz[1], y_off)\n            for cls_id, sc, bx in zip(labels, scores, boxes):\n                cid_i = int(cls_id)\n                sc_f = max(0.0, min(1.0, float(sc)))\n                x1, y1, x2, y2 = float(bx[0]), float(bx[1]), float(bx[2]), float(bx[3])\n                if 0 <= cid_i <= 3 and math.isfinite(sc_f) and (x2 - x1) >= 0.5 and (y2 - y1) >= 0.5:\n                    rows.append([iid, cid_i, f\'{sc_f:.6f}\', f\'{x1:.2f}\', f\'{y1:.2f}\', f\'{x2:.2f}\', f\'{y2:.2f}\'])\n    with out_csv.open(\'w\', newline=\'\', encoding=\'utf-8\') as f:\n        w = csv.writer(f)\n        w.writerow([\'image_id\', \'class_id\', \'score\', \'x_min\', \'y_min\', \'x_max\', \'y_max\'])\n        w.writerows(rows)\n\ndef _run_official_score_py(pred_csv: Path, score_json: Path, speed_json: Path = None) -> dict:\n    cmd = [\n        sys.executable, str(KIT_LOCAL / \'score.py\'),\n        \'--pred\', str(pred_csv),\n        \'--gt\', str(KIT_LOCAL / \'val_labels.csv\'),\n        \'--image-ids\', str(KIT_LOCAL / \'val_image_ids.csv\'),\n        \'--json-out\', str(score_json),\n    ]\n    if speed_json is not None and speed_json.is_file():\n        cmd.extend([\'--speed-json\', str(speed_json)])\n    res = subprocess.run(cmd, text=True, capture_output=True)\n    if res.returncode != 0:\n        if res.stdout:\n            print(\'score.py stdout:\\n\', res.stdout)\n        if res.stderr:\n            print(\'score.py stderr:\\n\', res.stderr)\n        res.check_returncode()\n    return json.loads(score_json.read_text(encoding=\'utf-8\'))\n\nmodel = DFineForObjectDetection.from_pretrained(\n    MODEL_REPO,\n    id2label=CLASS_MAP,\n    label2id=LABEL2ID,\n    ignore_mismatched_sizes=True,\n    cache_dir=str(HF_CACHE_DIR),\n    **_TF_DTYPE_KW,\n).to(DEVICE)\n\nlr_head = float(EXP_CFG.get(\'lr_head\', 2e-4))\nlr_backbone = float(EXP_CFG.get(\'lr_backbone\', 2e-5))\nweight_decay = float(EXP_CFG.get(\'weight_decay\', 1e-4))\nmax_grad_norm = float(EXP_CFG.get(\'max_grad_norm\', 0.1))\nema_decay = float(EXP_CFG.get(\'ema_decay\', 0.9998))\neval_every = int(EXP_CFG.get(\'eval_every_epochs\', 3))\nconf_floor = float(EXP_CFG.get(\'conf_floor\', 0.001))\nmax_det = int(EXP_CFG.get(\'max_det\', 100))\n\nparam_groups = [\n    {\'params\': [p for n, p in model.named_parameters() if \'backbone\' in n and p.requires_grad], \'lr\': lr_backbone, \'weight_decay\': weight_decay},\n    {\'params\': [p for n, p in model.named_parameters() if \'backbone\' not in n and p.requires_grad], \'lr\': lr_head, \'weight_decay\': weight_decay},\n]\noptimizer = torch.optim.AdamW(param_groups, betas=(0.9, 0.999))\nsteps_per_epoch = max(1, math.ceil(len(train_loader) / GRAD_ACCUM))\ntotal_opt_steps = max(1, TARGET_EPOCHS * steps_per_epoch)\nwarmup_steps = min(total_opt_steps // 2, int(round(float(EXP_CFG.get(\'warmup_epochs\', 1.5)) * steps_per_epoch)))\nscheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_opt_steps)\nscaler = torch.amp.GradScaler(\'cuda\', enabled=(AMP_DTYPE == torch.float16))\nema = ModelEMA(model, decay=ema_decay)\n\nBEST_DIR_LOCAL = WORK_ROOT / \'dfine_m_best\'\nBEST_DIR_DRIVE = RUN_DRIVE_DIR / \'weights\' / \'dfine_m_best\'\nLAST_CKPT_LOCAL = WORK_ROOT / \'last_resume_ckpt.pt\'\nLAST_CKPT_DRIVE = RUN_DRIVE_DIR / \'checkpoints\' / \'last_resume_ckpt.pt\'\nRESULTS_CSV_LOCAL = WORK_ROOT / \'results.csv\'\nRESULTS_CSV_DRIVE = RUN_DRIVE_DIR / \'weights\' / \'results.csv\'\n\nstart_epoch = 1\nbest_acc = -1.0\nhistory_rows = []\nnonfinite_skips = 0\n\nif LAST_CKPT_DRIVE.is_file():\n    print(\'Found resumable checkpoint, restoring state from:\', LAST_CKPT_DRIVE)\n    ckpt = torch.load(LAST_CKPT_DRIVE, map_location=DEVICE, weights_only=False)\n    model.load_state_dict(ckpt[\'model\'])\n    ema.shadow = {k: v.to(DEVICE) for k, v in ckpt[\'ema_shadow\'].items()}\n    ema.updates = int(ckpt[\'ema_updates\'])\n    optimizer.load_state_dict(ckpt[\'optimizer\'])\n    scheduler.load_state_dict(ckpt[\'scheduler\'])\n    if \'scaler\' in ckpt and ckpt[\'scaler\'] is not None:\n        scaler.load_state_dict(ckpt[\'scaler\'])\n    start_epoch = int(ckpt[\'epoch\']) + 1\n    best_acc = float(ckpt.get(\'best_acc\', -1.0))\n    history_rows = list(ckpt.get(\'history\', []))\n\nt_train_start = time.monotonic()\nfor epoch in range(start_epoch, TARGET_EPOCHS + 1):\n    t_ep = time.monotonic()\n    train_sampler.set_epoch(epoch)\n    model.train()\n    optimizer.zero_grad(set_to_none=True)\n    ep_loss_sum, ep_steps = 0.0, 0\n\n    for step_idx, batch in enumerate(train_loader, start=1):\n        px = batch[\'pixel_values\'].to(DEVICE, non_blocking=True)\n        lbls = [{k: v.to(DEVICE, non_blocking=True) for k, v in t.items()} for t in batch[\'labels\']]\n        with torch.amp.autocast(\'cuda\', dtype=AMP_DTYPE, enabled=(AMP_DTYPE is not None)):\n            out = model(pixel_values=px, labels=lbls)\n            raw_loss = out.loss\n            loss = raw_loss / float(GRAD_ACCUM)\n\n        if not torch.isfinite(raw_loss):\n            nonfinite_skips += 1\n            optimizer.zero_grad(set_to_none=True)\n            continue\n\n        if scaler.is_enabled():\n            scaler.scale(loss).backward()\n        else:\n            loss.backward()\n\n        ep_loss_sum += float(raw_loss.detach().item())\n        ep_steps += 1\n\n        if (step_idx % GRAD_ACCUM == 0) or (step_idx == len(train_loader)):\n            if scaler.is_enabled():\n                scaler.unscale_(optimizer)\n            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)\n            if torch.isfinite(grad_norm):\n                if scaler.is_enabled():\n                    scaler.step(optimizer)\n                    scaler.update()\n                else:\n                    optimizer.step()\n                scheduler.step()\n                ema.update(model)\n            else:\n                nonfinite_skips += 1\n                if scaler.is_enabled():\n                    scaler.update()\n            optimizer.zero_grad(set_to_none=True)\n\n    mean_loss = ep_loss_sum / max(1, ep_steps)\n    cur_lr = float(optimizer.param_groups[1][\'lr\'])\n    ep_row = {\n        \'epoch\': epoch,\n        \'train_loss\': round(mean_loss, 5),\n        \'lr_head\': cur_lr,\n        \'lr_backbone\': float(optimizer.param_groups[0][\'lr\']),\n        \'elapsed_s\': round(time.monotonic() - t_train_start, 2),\n        \'epoch_s\': round(time.monotonic() - t_ep, 2),\n        \'nonfinite_skips\': int(nonfinite_skips),\n        \'val_accuracy_score\': None,\n        \'val_mAP50\': None,\n        \'val_mAP50_95\': None,\n        \'val_P\': None,\n        \'val_R\': None,\n    }\n\n    do_eval = (epoch % eval_every == 0) or (epoch >= max(1, TARGET_EPOCHS - 2)) or (epoch == 1)\n    if do_eval:\n        ema.apply_shadow(model)\n        val_csv_ep = WORK_ROOT / f\'val_pred_ep{epoch}.csv\'\n        val_json_ep = WORK_ROOT / f\'val_score_ep{epoch}.json\'\n        _evaluate_dfine_to_csv(model, val_loader, val_csv_ep, conf_floor=conf_floor, max_det=max_det)\n        scr_ep = _run_official_score_py(val_csv_ep, val_json_ep)\n        acc_ep = float(scr_ep[\'combined\'][\'accuracy_score\'])\n        m_ep = scr_ep[\'main\']\n        ep_row.update({\n            \'val_accuracy_score\': round(acc_ep, 4),\n            \'val_mAP50\': round(float(m_ep[\'mAP50\']), 5),\n            \'val_mAP50_95\': round(float(m_ep[\'mAP50_95\']), 5),\n            \'val_P\': round(float(m_ep[\'precision\']), 5),\n            \'val_R\': round(float(m_ep[\'recall\']), 5),\n        })\n        print(f\'Epoch {epoch:02d}/{TARGET_EPOCHS} | loss={mean_loss:.4f} | lr={cur_lr:.2e} | Val Acc={acc_ep:.2f} | mAP50={m_ep["mAP50"]:.4f} | mAP50:95={m_ep["mAP50_95"]:.4f} | P={m_ep["precision"]:.4f} R={m_ep["recall"]:.4f} | {ep_row["epoch_s"]:.1f}s\')\n\n        if acc_ep > best_acc:\n            best_acc = acc_ep\n            if BEST_DIR_LOCAL.exists():\n                shutil.rmtree(BEST_DIR_LOCAL)\n            BEST_DIR_LOCAL.mkdir(parents=True, exist_ok=True)\n            model.save_pretrained(str(BEST_DIR_LOCAL))\n            IMAGE_PROCESSOR.save_pretrained(str(BEST_DIR_LOCAL))\n            if BEST_DIR_DRIVE.exists():\n                shutil.rmtree(BEST_DIR_DRIVE)\n            shutil.copytree(BEST_DIR_LOCAL, BEST_DIR_DRIVE)\n            print(f\'  -> Saved NEW BEST D-FINE-M EMA weights (Acc={best_acc:.2f}) to Drive.\')\n        ema.restore(model)\n    else:\n        print(f\'Epoch {epoch:02d}/{TARGET_EPOCHS} | loss={mean_loss:.4f} | lr={cur_lr:.2e} | {ep_row["epoch_s"]:.1f}s (eval skipped)\')\n\n    history_rows.append(ep_row)\n    pd.DataFrame(history_rows).to_csv(RESULTS_CSV_LOCAL, index=False)\n    shutil.copy2(RESULTS_CSV_LOCAL, RESULTS_CSV_DRIVE)\n\n    # Save full resumable checkpoint periodically + at final epoch\n    if (epoch % 3 == 0) or (epoch == TARGET_EPOCHS):\n        ckpt_payload = {\n            \'epoch\': epoch,\n            \'best_acc\': best_acc,\n            \'model\': model.state_dict(),\n            \'ema_shadow\': ema.shadow,\n            \'ema_updates\': ema.updates,\n            \'optimizer\': optimizer.state_dict(),\n            \'scheduler\': scheduler.state_dict(),\n            \'scaler\': scaler.state_dict() if scaler.is_enabled() else None,\n            \'history\': history_rows,\n            \'config\': EXP_CFG,\n        }\n        torch.save(ckpt_payload, LAST_CKPT_LOCAL)\n        shutil.copy2(LAST_CKPT_LOCAL, LAST_CKPT_DRIVE)\n    check_budget()\n\nif not BEST_DIR_LOCAL.is_dir():\n    raise FileNotFoundError(f\'Missing best D-FINE-M directory at {BEST_DIR_LOCAL}\')\n\nREPO_DIR = WORK_ROOT / \'submission_repo\'\n(REPO_DIR / \'weights\').mkdir(parents=True, exist_ok=True)\nif (REPO_DIR / \'weights\' / \'dfine_m_best\').exists():\n    shutil.rmtree(REPO_DIR / \'weights\' / \'dfine_m_best\')\nshutil.copytree(BEST_DIR_LOCAL, REPO_DIR / \'weights\' / \'dfine_m_best\')\n\nckpt_rel_files = [\n    f\'03_results/runs/{RUN_ID}/weights/dfine_m_best/model.safetensors\',\n    f\'03_results/runs/{RUN_ID}/weights/dfine_m_best/config.json\',\n    f\'03_results/runs/{RUN_ID}/weights/results.csv\',\n]\nmanifest_best = autopilot.save_checkpoint_manifest(PROJECT_ROOT, RUN_ID, \'best_model\', CHECKPOINT_CONFIG, ckpt_rel_files)\nresume_rel_files = ckpt_rel_files + ([f\'03_results/runs/{RUN_ID}/checkpoints/last_resume_ckpt.pt\'] if LAST_CKPT_DRIVE.is_file() else [])\nmanifest_resume = autopilot.save_checkpoint_manifest(PROJECT_ROOT, RUN_ID, \'last_resume\', CHECKPOINT_CONFIG, resume_rel_files)\nTASK_ARTIFACTS[\'best_weights_dir\'] = f\'03_results/runs/{RUN_ID}/weights/dfine_m_best\'\nTASK_ARTIFACTS[\'checkpoint_manifest\'] = str(manifest_best.relative_to(PROJECT_ROOT))\nTASK_ARTIFACTS[\'resume_manifest\'] = str(manifest_resume.relative_to(PROJECT_ROOT))\nTASK_ARTIFACTS[\'gpu_tier\'] = GPU_TIER\nprint(\'Saved best EMA checkpoint & full resume manifest.\')')


## 5. 4-Way Inference A/B Test (LUT / Bright-CLAHE / Flip-TTA+WBF) & Speed Benchmark

Benchmark raw Sky-Crop vs Gamma LUT (<60) vs LUT + Bright-Concrete CLAHE (>155) vs Batched Horizontal Flip-TTA + Class-Aware WBF on official benchmark_speed.py + score.py and export ensemble val_pred_dfine_m.csv.


In [ ]:
_execute_stage(4, 'luma_thresh = float(EXP_CFG.get(\'lowlight_luma_thresh\', 60.0))\ntarget_luma = float(EXP_CFG.get(\'lowlight_target_luma\', 115.0))\nbright_clahe_thresh = float(EXP_CFG.get(\'bright_clahe_luma_thresh\', 155.0))\nwbf_iou_thr = float(EXP_CFG.get(\'wbf_iou_thr\', 0.55))\n\ndef _make_dfine_inference_py(enable_lut: bool, enable_bright_clahe: bool, enable_flip_tta: bool) -> str:\n    return f\'\'\'import math\nimport os\nimport cv2\nimport numpy as np\nimport torch\nimport transformers\nfrom transformers import AutoImageProcessor, DFineForObjectDetection\n\n_TF_MAJOR = int(transformers.__version__.split(".")[0])\n_TF_DTYPE_KW = {{"dtype": torch.float32}} if _TF_MAJOR >= 5 else {{"torch_dtype": torch.float32}}\n\n_HERE = os.path.dirname(os.path.abspath(__file__))\nWEIGHTS_DIR = os.path.join(_HERE, "weights", "dfine_m_best")\nIMGSZ = {IMGSZ}\nSKY_CROP_FRAC = {SKY_CROP_FRAC}\nCONF_FLOOR = {conf_floor}\nMAX_DET = {max_det}\nENABLE_LUT = {enable_lut!r}\nENABLE_BRIGHT_CLAHE = {enable_bright_clahe!r}\nENABLE_FLIP_TTA = {enable_flip_tta!r}\nLUMA_THRESH = {luma_thresh}\nTARGET_LUMA = {target_luma}\nBRIGHT_CLAHE_THRESH = {bright_clahe_thresh}\nWBF_IOU_THR = {wbf_iou_thr}\n\ndef _build_lut():\n    x = np.arange(256, dtype=np.float32) / 255.0\n    tgt = math.log(TARGET_LUMA / 255.0)\n    luts = {{}}\n    for lum in range(int(LUMA_THRESH)):\n        gamma = max(0.35, min(0.85, tgt / math.log(max(float(lum), 8.0) / 255.0)))\n        luts[lum] = np.clip(np.round(255.0 * (x ** gamma)), 0, 255).astype(np.uint8)\n    return luts\n\ndef _apply_clahe_rgb(img_rgb):\n    lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)\n    l_ch, a_ch, b_ch = cv2.split(lab)\n    l_eq = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(l_ch)\n    return cv2.cvtColor(cv2.merge((l_eq, a_ch, b_ch)), cv2.COLOR_LAB2RGB)\n\ndef _wbf_two_views(ba, sa, la, bb, sb, lb, iou_thr, max_det):\n    out = []\n    classes = sorted(set([int(c) for c in la] + [int(c) for c in lb]))\n    for cid in classes:\n        cands = []\n        for b, s, c in zip(ba, sa, la):\n            if int(c) == cid:\n                cands.append((float(s), np.asarray(b, dtype=np.float64)))\n        for b, s, c in zip(bb, sb, lb):\n            if int(c) == cid:\n                cands.append((float(s), np.asarray(b, dtype=np.float64)))\n        cands.sort(key=lambda t: t[0], reverse=True)\n        clusters = []\n        for sc, box in cands:\n            best_i, best_iou = -1, iou_thr\n            bx1, by1, bx2, by2 = box\n            barea = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)\n            for ci, cl in enumerate(clusters):\n                cx1, cy1, cx2, cy2 = cl["fused"]\n                iw = max(0.0, min(bx2, cx2) - max(bx1, cx1))\n                ih = max(0.0, min(by2, cy2) - max(by1, cy1))\n                inter = iw * ih\n                if inter <= 0.0:\n                    continue\n                carea = max(0.0, cx2 - cx1) * max(0.0, cy2 - cy1)\n                iou = inter / max(1e-9, barea + carea - inter)\n                if iou >= best_iou:\n                    best_iou, best_i = iou, ci\n            if best_i >= 0:\n                cl = clusters[best_i]\n                ws = cl["w_sum"] + sc\n                cl["fused"] = (cl["fused"] * cl["w_sum"] + box * sc) / max(1e-9, ws)\n                cl["w_sum"] = ws\n                cl["scores"].append(sc)\n            else:\n                clusters.append({{"fused": box.copy(), "w_sum": sc, "scores": [sc]}})\n        for cl in clusters:\n            fb = cl["fused"]\n            out.append((cid, float(sum(cl["scores"]) / 2.0), float(fb[0]), float(fb[1]), float(fb[2]), float(fb[3])))\n    out.sort(key=lambda t: t[1], reverse=True)\n    return out[:max_det]\n\nclass _DFineHolder:\n    def __init__(self, model, processor, device, amp_dtype, luts):\n        self.model = model\n        self.processor = processor\n        self.device = device\n        self.amp_dtype = amp_dtype\n        self.luts = luts\n\ndef load(device="cuda"):\n    dev = torch.device(device if (torch.cuda.is_available() and str(device).startswith("cuda")) else "cpu")\n    proc = AutoImageProcessor.from_pretrained(WEIGHTS_DIR, local_files_only=True)\n    net = DFineForObjectDetection.from_pretrained(WEIGHTS_DIR, local_files_only=True, **_TF_DTYPE_KW).to(dev).eval()\n    amp_dt = torch.bfloat16 if (dev.type == "cuda" and torch.cuda.is_bf16_supported()) else (torch.float16 if dev.type == "cuda" else None)\n    luts = _build_lut() if ENABLE_LUT else None\n    return _DFineHolder(net, proc, dev, amp_dt, luts)\n\n@torch.no_grad()\ndef predict(model, image):\n    orig_h, orig_w = int(image.shape[0]), int(image.shape[1])\n    y_off = max(0, min(orig_h - 32, int(round(orig_h * SKY_CROP_FRAC))))\n    crop = image[y_off:, :, :]\n    if model.luts is not None or ENABLE_BRIGHT_CLAHE:\n        mean_lum = float(crop[::4, ::4].mean())\n        if model.luts is not None and mean_lum < LUMA_THRESH:\n            crop = model.luts[max(0, min(int(LUMA_THRESH) - 1, int(mean_lum)))][crop]\n        elif ENABLE_BRIGHT_CLAHE and mean_lum > BRIGHT_CLAHE_THRESH:\n            crop = _apply_clahe_rgb(crop)\n\n    crop_h, crop_w = int(crop.shape[0]), int(crop.shape[1])\n    rz = cv2.resize(crop, (IMGSZ, IMGSZ), interpolation=cv2.INTER_LINEAR)\n    t1 = torch.from_numpy(np.ascontiguousarray(rz)).permute(2, 0, 1).to(torch.float32).div_(255.0)\n    if ENABLE_FLIP_TTA:\n        t2 = torch.flip(t1, dims=[2])\n        px = torch.stack([t1, t2], dim=0).to(model.device)\n        t_sizes = torch.tensor([[crop_h, crop_w], [crop_h, crop_w]], dtype=torch.long)\n    else:\n        px = t1.unsqueeze(0).to(model.device)\n        t_sizes = torch.tensor([[crop_h, crop_w]], dtype=torch.long)\n\n    with torch.amp.autocast("cuda", dtype=model.amp_dtype, enabled=(model.device.type == "cuda" and model.amp_dtype is not None)):\n        outs = model.model(pixel_values=px)\n    outs.logits = outs.logits.float()\n    outs.pred_boxes = outs.pred_boxes.float()\n    decoded = model.processor.post_process_object_detection(outs, target_sizes=t_sizes, threshold=CONF_FLOOR)\n\n    r0 = decoded[0]\n    b0 = r0["boxes"].detach().float().cpu().numpy()\n    s0 = r0["scores"].detach().float().cpu().numpy()\n    l0 = r0["labels"].detach().cpu().numpy().astype(int)\n\n    if ENABLE_FLIP_TTA and len(decoded) > 1:\n        r1 = decoded[1]\n        b1 = r1["boxes"].detach().float().cpu().numpy()\n        s1 = r1["scores"].detach().float().cpu().numpy()\n        l1 = r1["labels"].detach().cpu().numpy().astype(int)\n        if len(b1) > 0:\n            b1_u = b1.copy()\n            b1_u[:, 0] = float(crop_w) - b1[:, 2]\n            b1_u[:, 2] = float(crop_w) - b1[:, 0]\n        else:\n            b1_u = b1\n        preds = _wbf_two_views(b0, s0, l0, b1_u, s1, l1, WBF_IOU_THR, MAX_DET)\n        out_list = []\n        for cid, sc, x1, y1, x2, y2 in preds:\n            cid_i = int(cid)\n            sc_f = max(0.0, min(1.0, float(sc)))\n            fx1 = max(0.0, min(float(orig_w), float(x1)))\n            fy1 = max(0.0, min(float(orig_h), float(y1) + float(y_off)))\n            fx2 = max(0.0, min(float(orig_w), float(x2)))\n            fy2 = max(0.0, min(float(orig_h), float(y2) + float(y_off)))\n            if 0 <= cid_i <= 3 and math.isfinite(sc_f) and (fx2 - fx1) >= 0.5 and (fy2 - fy1) >= 0.5:\n                out_list.append((cid_i, sc_f, fx1, fy1, fx2, fy2))\n        return out_list\n\n    if len(s0) > MAX_DET:\n        keep = np.argsort(-s0)[:MAX_DET]\n        s0, l0, b0 = s0[keep], l0[keep], b0[keep]\n    out_list = []\n    for cid, sc, bx in zip(l0, s0, b0):\n        cid_i = int(cid)\n        sc_f = max(0.0, min(1.0, float(sc)))\n        fx1 = max(0.0, min(float(orig_w), float(bx[0])))\n        fy1 = max(0.0, min(float(orig_h), float(bx[1]) + float(y_off)))\n        fx2 = max(0.0, min(float(orig_w), float(bx[2])))\n        fy2 = max(0.0, min(float(orig_h), float(bx[3]) + float(y_off)))\n        if 0 <= cid_i <= 3 and math.isfinite(sc_f) and (fx2 - fx1) >= 0.5 and (fy2 - fy1) >= 0.5:\n            out_list.append((cid_i, sc_f, fx1, fy1, fx2, fy2))\n    return out_list\n\'\'\'\n\ndef _eval_dfine_variant(tag: str, enable_lut: bool, enable_bright_clahe: bool, enable_flip_tta: bool) -> dict:\n    code_str = _make_dfine_inference_py(enable_lut, enable_bright_clahe, enable_flip_tta)\n    (REPO_DIR / \'inference.py\').write_text(code_str, encoding=\'utf-8\')\n    v_pred = WORK_ROOT / f\'{tag}_val_pred.csv\'\n    v_spd = WORK_ROOT / f\'{tag}_val_speed.json\'\n    v_scr = WORK_ROOT / f\'{tag}_val_score.json\'\n    s_pred = WORK_ROOT / f\'{tag}_stress_pred.csv\'\n    s_spd = WORK_ROOT / f\'{tag}_stress_speed.json\'\n    s_scr = WORK_ROOT / f\'{tag}_stress_score.json\'\n    subprocess.run([sys.executable, str(KIT_LOCAL / \'benchmark_speed.py\'), \'--repo\', str(REPO_DIR), \'--images\', str(KIT_LOCAL / \'val_images\'), \'--out-csv\', str(v_pred), \'--out-json\', str(v_spd), \'--runs\', \'2\', \'--repeat-images\', \'300\'], check=True)\n    subprocess.run([sys.executable, str(KIT_LOCAL / \'score.py\'), \'--pred\', str(v_pred), \'--gt\', str(KIT_LOCAL / \'val_labels.csv\'), \'--image-ids\', str(KIT_LOCAL / \'val_image_ids.csv\'), \'--speed-json\', str(v_spd), \'--json-out\', str(v_scr)], check=True)\n    subprocess.run([sys.executable, str(KIT_LOCAL / \'benchmark_speed.py\'), \'--repo\', str(REPO_DIR), \'--images\', str(VAL_DEGRADED_DIR), \'--out-csv\', str(s_pred), \'--out-json\', str(s_spd), \'--runs\', \'1\', \'--warmup\', \'5\'], check=True)\n    subprocess.run([sys.executable, str(KIT_LOCAL / \'score.py\'), \'--pred\', str(s_pred), \'--gt\', str(KIT_LOCAL / \'val_labels.csv\'), \'--image-ids\', str(KIT_LOCAL / \'val_image_ids.csv\'), \'--json-out\', str(s_scr)], check=True)\n    spd = json.loads(v_spd.read_text(encoding=\'utf-8\'))\n    scr = json.loads(v_scr.read_text(encoding=\'utf-8\'))\n    st_scr = json.loads(s_scr.read_text(encoding=\'utf-8\'))\n    m = scr[\'main\']\n    rob = float(st_scr[\'main\'][\'mAP50\'])\n    spd_sc = float(spd[\'speed_score\'])\n    final_sc = 100.0 * (0.30 * float(m[\'mAP50\']) + 0.25 * float(m[\'mAP50_95\']) + 0.15 * float(m[\'precision\']) + 0.15 * float(m[\'recall\']) + 0.10 * spd_sc + 0.05 * rob)\n    return {\n        \'tag\': tag,\n        \'enable_lut\': enable_lut,\n        \'enable_bright_clahe\': enable_bright_clahe,\n        \'enable_flip_tta\': enable_flip_tta,\n        \'code\': code_str,\n        \'val_pred_path\': v_pred,\n        \'val_speed\': spd,\n        \'val_score\': scr,\n        \'stress_score\': st_scr,\n        \'final_score_proxy_100\': final_sc,\n        \'robustness_mAP50_synth\': rob,\n        \'v_spd_path\': v_spd,\n        \'v_scr_path\': v_scr,\n        \'s_scr_path\': s_scr,\n    }\n\nres_a = _eval_dfine_variant(\'A_sky_raw\', False, False, False)\nres_b = _eval_dfine_variant(\'B_sky_lut\', True, False, False)\nres_c = _eval_dfine_variant(\'C_sky_lut_clahe\', True, True, False)\nbest_pre = max([res_a, res_b, res_c], key=lambda r: r[\'final_score_proxy_100\'])\ncandidates = [res_a, res_b, res_c]\nif bool(EXP_CFG.get(\'test_flip_tta\', True)):\n    res_d = _eval_dfine_variant(\'D_sky_bestpre_flip_tta_wbf\', best_pre[\'enable_lut\'], best_pre[\'enable_bright_clahe\'], True)\n    candidates.append(res_d)\n\nwinner = max(candidates, key=lambda r: r[\'final_score_proxy_100\'])\nab_delta = float(winner[\'final_score_proxy_100\'] - res_a[\'final_score_proxy_100\'])\n\n(REPO_DIR / \'inference.py\').write_text(winner[\'code\'], encoding=\'utf-8\')\nshutil.copy2(REPO_DIR / \'inference.py\', RUN_DRIVE_DIR / \'inference.py\')\nshutil.copy2(winner[\'v_spd_path\'], RUN_DRIVE_DIR / \'val_speed.json\')\nshutil.copy2(winner[\'v_scr_path\'], RUN_DRIVE_DIR / \'val_score.json\')\nshutil.copy2(winner[\'s_scr_path\'], RUN_DRIVE_DIR / \'stress_score.json\')\nshutil.copy2(winner[\'val_pred_path\'], RUN_DRIVE_DIR / \'val_pred_dfine_m.csv\')\n\nab_summary = {\n    c[\'tag\']: {\n        \'accuracy_score_100\': c[\'val_score\'][\'combined\'][\'accuracy_score\'],\n        \'mAP50\': c[\'val_score\'][\'main\'][\'mAP50\'],\n        \'mAP50_95\': c[\'val_score\'][\'main\'][\'mAP50_95\'],\n        \'robustness_mAP50\': c[\'robustness_mAP50_synth\'],\n        \'median_ms\': c[\'val_speed\'][\'median_ms\'],\n        \'final_score_proxy_100\': round(c[\'final_score_proxy_100\'], 4),\n    }\n    for c in candidates\n}\nab_summary[\'winner_tag\'] = winner[\'tag\']\nab_summary[\'ab_winner_delta_final_pts\'] = round(ab_delta, 4)\n(RUN_DRIVE_DIR / \'ab_inference_comparison.json\').write_text(json.dumps(ab_summary, indent=2), encoding=\'utf-8\')\nprint(\'4-Way D-FINE-M Inference A/B Comparison:\', json.dumps(ab_summary, indent=2))\n\nm = winner[\'val_score\'][\'main\']\npc = m[\'per_class\']\ntelemetry = autopilot.collect_colab_telemetry(time.monotonic() - STARTED, burn_rate_cu_per_hour=CONFIG.get(\'budget\', {}).get(\'burn_rate_cu_per_hour\'))\nmetrics.update({\n    \'accuracy_score_100\': float(winner[\'val_score\'][\'combined\'][\'accuracy_score\']),\n    \'mAP50\': float(m[\'mAP50\']),\n    \'mAP50_95\': float(m[\'mAP50_95\']),\n    \'precision\': float(m[\'precision\']),\n    \'recall\': float(m[\'recall\']),\n    \'ap50_D00\': float(pc[\'D00_longitudinal_crack\'][\'AP50\']),\n    \'ap50_D10\': float(pc[\'D10_transverse_crack\'][\'AP50\']),\n    \'ap50_D20\': float(pc[\'D20_alligator_crack\'][\'AP50\']),\n    \'ap50_D40\': float(pc[\'D40_pothole\'][\'AP50\']),\n    \'speed_score_t4\': float(winner[\'val_speed\'][\'speed_score\']),\n    \'median_ms_t4\': float(winner[\'val_speed\'][\'median_ms\']),\n    \'robustness_mAP50_synth\': float(winner[\'robustness_mAP50_synth\']),\n    \'final_score_proxy_100\': float(winner[\'final_score_proxy_100\']),\n    \'ab_winner_delta_final_pts\': float(ab_delta),\n    \'train_images_count\': int(len(train_records)),\n    \'val_images_count\': int(len(val_records)),\n    \'effective_batch_size\': int(BATCH_SIZE * GRAD_ACCUM),\n})\nif \'peak_vram_gib\' in telemetry:\n    metrics[\'peak_vram_gib\'] = float(telemetry[\'peak_vram_gib\'])\nif \'estimated_compute_units\' in telemetry:\n    metrics[\'estimated_compute_units\'] = float(telemetry[\'estimated_compute_units\'])\n\nTASK_ARTIFACTS.update({\n    \'val_score_json\': f\'03_results/runs/{RUN_ID}/val_score.json\',\n    \'val_speed_json\': f\'03_results/runs/{RUN_ID}/val_speed.json\',\n    \'stress_score_json\': f\'03_results/runs/{RUN_ID}/stress_score.json\',\n    \'val_pred_dfine_m_csv\': f\'03_results/runs/{RUN_ID}/val_pred_dfine_m.csv\',\n    \'ab_inference_comparison\': f\'03_results/runs/{RUN_ID}/ab_inference_comparison.json\',\n    \'inference_py\': f\'03_results/runs/{RUN_ID}/inference.py\',\n    \'winner_inference_variant\': winner[\'tag\'],\n})\ncheck_budget()')


## 6. Full 4,200 Test Inference, Ensemble CSV Export & Submission Validation

Run winning inference.py over all 4,200 test images, export ensemble-ready test_pred_dfine_m.csv, generate and validate submission CSV with check_submission.py.


In [ ]:
_execute_stage(5, "expected_test_ids = len(test_ids_df)\nactual_test_imgs = len(list((KIT_LOCAL / 'test_images').glob('*.jpg')))\nif actual_test_imgs < expected_test_ids:\n    raise RuntimeError(f'Expected {expected_test_ids} test images, got {actual_test_imgs}')\n\ntest_pred_csv = WORK_ROOT / 'test_pred_dfine_m.csv'\ntest_speed_json = WORK_ROOT / 'test_speed.json'\nsub_csv = WORK_ROOT / f'submission_{RUN_ID}.csv'\n\nsubprocess.run([sys.executable, str(KIT_LOCAL / 'benchmark_speed.py'), '--repo', str(REPO_DIR), '--images', str(KIT_LOCAL / 'test_images'), '--out-csv', str(test_pred_csv), '--out-json', str(test_speed_json), '--runs', '1', '--warmup', '10'], check=True)\nsubprocess.run([sys.executable, str(KIT_LOCAL / 'make_kaggle_submission.py'), '--pred', str(test_pred_csv), '--image-ids', str(KIT_LOCAL / 'test_image_ids.csv'), '--out', str(sub_csv)], check=True)\nsubprocess.run([sys.executable, str(KIT_LOCAL / 'check_submission.py'), '--pred', str(sub_csv), '--image-ids', str(KIT_LOCAL / 'test_image_ids.csv')], check=True)\n\ndrive_sub_dir = PROJECT_ROOT / '04_submissions'\ndrive_sub_dir.mkdir(parents=True, exist_ok=True)\ndrive_sub = drive_sub_dir / f'submission_{RUN_ID}.csv'\nshutil.copy2(sub_csv, drive_sub)\nshutil.copy2(test_pred_csv, RUN_DRIVE_DIR / 'test_pred_dfine_m.csv')\nshutil.copy2(test_speed_json, RUN_DRIVE_DIR / 'test_speed.json')\n\nTASK_ARTIFACTS['test_pred_dfine_m_csv'] = f'03_results/runs/{RUN_ID}/test_pred_dfine_m.csv'\nTASK_ARTIFACTS['test_speed_json'] = f'03_results/runs/{RUN_ID}/test_speed.json'\nTASK_ARTIFACTS['submission_csv'] = f'04_submissions/submission_{RUN_ID}.csv'\nTASK_ARTIFACTS['submission_sha256'] = autopilot.file_digest(drive_sub)\nTASK_ARTIFACTS['submission_status'] = 'validated_by_check_submission_py'\nprint('Validated D-FINE-M submission saved to:', drive_sub)\nprint('Ensemble-ready predictions saved to:', RUN_DRIVE_DIR / 'val_pred_dfine_m.csv', 'and', RUN_DRIVE_DIR / 'test_pred_dfine_m.csv')")


## 7. Kaggle Export, Summary & Ensemble Hand-Off


In [ ]:
export_root = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else WORK_BASE
sub_rel = TASK_ARTIFACTS.get('submission_csv')
if not sub_rel:
    raise RuntimeError('Missing submission_csv in TASK_ARTIFACTS')
sub_src = PROJECT_ROOT / sub_rel
if not sub_src.is_file():
    raise FileNotFoundError(f'Missing validated submission at {sub_src}')

kaggle_sub = export_root / 'submission.csv'
shutil.copy2(sub_src, kaggle_sub)

for pred_key, dst_name in (
    ('val_pred_dfine_m_csv', 'val_pred_dfine_m.csv'),
    ('test_pred_dfine_m_csv', 'test_pred_dfine_m.csv'),
):
    rel = TASK_ARTIFACTS.get(pred_key)
    if rel and (PROJECT_ROOT / rel).is_file():
        shutil.copy2(PROJECT_ROOT / rel, export_root / dst_name)

run_summary = {
    'run_id': RUN_ID,
    'runtime': 'kaggle',
    'metrics': metrics,
    'artifacts': TASK_ARTIFACTS,
    'config': CONFIG,
}
(RUN_DRIVE_DIR / 'run_summary.json').write_text(json.dumps(run_summary, indent=2), encoding='utf-8')
shutil.copy2(RUN_DRIVE_DIR / 'run_summary.json', export_root / 'run_summary.json')

print('=== Kaggle D-FINE-M Run Summary ===')
print(json.dumps({'run_id': RUN_ID, 'metrics': metrics, 'artifacts': TASK_ARTIFACTS}, indent=2))
print(f'Ready for one-click Submit: {kaggle_sub}')
print(f'Ensemble CSVs saved to: {export_root / "val_pred_dfine_m.csv"} & {export_root / "test_pred_dfine_m.csv"}')
